# XGBoost Nutrient-Aware Pathogen Suppression Predictor 0201

Run02

In [2]:

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge, LassoCV, ElasticNetCV
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, GroupShuffleSplit, GroupKFold
from scipy.stats import pearsonr, spearmanr


In [3]:

df = pd.read_csv(r"C:\Users\Pratham Shah\Downloads\Tubingen\Tabular_Models_Dataset\Updated02\community_index_v9.csv")

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nTarget (community_suppression) statistics:")
print(df['community_suppression'].describe().round(4))

print("\nRows per diversity level:")
print(df['diversity_level'].value_counts().sort_index())

print("\nRows per community size:")
print(df['community_size'].value_counts().sort_index())

Dataset shape: 48,000 rows × 168 columns

Target (community_suppression) statistics:
count    48000.0000
mean         0.6698
std          0.3767
min          0.0000
25%          0.4471
50%          0.8554
75%          0.9710
max          0.9990
Name: community_suppression, dtype: float64

Rows per diversity level:
diversity_level
1     15000
2     12000
4      9000
8      6000
16     6000
Name: count, dtype: int64

Rows per community size:
community_size
12    9600
16    9600
24    9600
32    9600
48    9600
Name: count, dtype: int64


In [4]:
# Species membership columns (100 binary features) 
# Each column is 1 if that species is present in the community, 0 otherwise.
species_cols = [f"Species_{i:04d}" for i in range(2, 102)]


# R0 nutrient concentration columns (20 features)
# R0_c1 through R0_c20: initial concentration of each carbon source.
# Most are 0 (inactive). Active ones are ~20/n_active
r0_cols = [f"R0_c{k}" for k in range(1, 21)]


# Active flags (20 binary features)
# active_c1 through active_c20: 1 if that carbon source is supplied, 0 if not.
# These are redundant with R0 (if R0>0 then active=1) but are kept anyways
# CLARIFY ONCE: DO WE NEED TO KEEP THIS?
active_cols = [f"active_c{k}" for k in range(1, 21)]

# Structural environment features (2 features)
# community_size:  how many species are in this community (12, 16, 24, 32, 48).
# diversity_level: how many carbon sources are active (1, 2, 4, 8, 16).

structural_cols = ['community_size', 'diversity_level']

# Combine into full feature matrix

feature_cols = species_cols + r0_cols + active_cols + structural_cols
print(f"Total features: {len(feature_cols)}")
print(f"  Species membership:   {len(species_cols)}")
print(f"  R0 concentrations:    {len(r0_cols)}")
print(f"  Active flags:         {len(active_cols)}")
print(f"  Structural (size/div):{len(structural_cols)}")


X = df[feature_cols]
y = df['community_suppression']

print(f"\nX shape: {X.shape}    y shape: {y.shape}")
print(f"y range: [{y.min():.4f}, {y.max():.4f}]   mean: {y.mean():.4f}")

Total features: 142
  Species membership:   100
  R0 concentrations:    20
  Active flags:         20
  Structural (size/div):2

X shape: (48000, 142)    y shape: (48000,)
y range: [0.0000, 0.9990]   mean: 0.6698


In [7]:
strat_key = df['community_size'].astype(str) + "_" + df['diversity_level'].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=strat_key
)


groups = df['broth_id'].values # used later for groupkfold

print(f"Train set: {X_train.shape[0]:,} rows")
print(f"Test set:  {X_test.shape[0]:,} rows")
print(f"Unique broths: {df['broth_id'].nunique()}")
print(f"\nDiversity level distribution in test set:")
print(df.loc[X_test.index, 'diversity_level'].value_counts().sort_index())

Train set: 38,400 rows
Test set:  9,600 rows
Unique broths: 32

Diversity level distribution in test set:
diversity_level
1     3000
2     2400
4     1800
8     1200
16    1200
Name: count, dtype: int64


In [8]:
# baseline models (lasso and elastic net added instead of just ridge)

from sklearn.linear_model import RidgeCV

print("Baseline comparisons")

# Baseline 1: Global mean predictor 
y_mean      = y_train.mean()
y_pred_mean = np.full_like(y_test, fill_value=y_mean, dtype=float)
r2_mean     = r2_score(y_test, y_pred_mean)
print(f"Baseline (global mean predictor):  R² = {r2_mean:.4f}")


scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Baseline 2: Ridge 
ridge_alphas = np.logspace(-3, 3, 100)

ridge_cv = RidgeCV(alphas=ridge_alphas, cv=5, scoring='r2')
ridge_cv.fit(X_train_scaled, y_train)

y_pred_ridge = ridge_cv.predict(X_test_scaled)
r2_ridge = r2_score(y_test, y_pred_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)

print(f"Ridge (alpha={ridge_cv.alpha_:.5f}):         R² = {r2_ridge:.4f}   MAE = {mae_ridge:.4f}")

# Baseline 3: Lasso with cross-validated alpha selection.
# LassoCV tries 50 alpha values on a log scale and picks the one with
# the lowest 5-fold CV error. max_iter=10000 prevents convergence warnings
# with 142 features.
lasso_cv       = LassoCV(cv=5, max_iter=10000, n_alphas=50, random_state=42, n_jobs=-1)
lasso_cv.fit(X_train_scaled, y_train)
y_pred_lasso   = lasso_cv.predict(X_test_scaled)
r2_lasso       = r2_score(y_test, y_pred_lasso)
mae_lasso      = mean_absolute_error(y_test, y_pred_lasso)
n_lasso_kept   = int(np.sum(lasso_cv.coef_ != 0))
print(f"Lasso (alpha={lasso_cv.alpha_:.5f}):       R² = {r2_lasso:.4f}   MAE = {mae_lasso:.4f}   features kept = {n_lasso_kept}/{len(feature_cols)}")

# Baseline 4: Elastic Net with cross-validated alpha and l1_ratio selection.
# l1_ratio controls the L1/L2 mix: 1.0 = pure Lasso, 0.0 = pure Ridge.
enet_cv        = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95],
                               cv=5, max_iter=10000, n_alphas=50,
                               random_state=42, n_jobs=-1)
enet_cv.fit(X_train_scaled, y_train)
y_pred_enet    = enet_cv.predict(X_test_scaled)
r2_enet        = r2_score(y_test, y_pred_enet)
mae_enet       = mean_absolute_error(y_test, y_pred_enet)
n_enet_kept    = int(np.sum(enet_cv.coef_ != 0))
print(f"ElasticNet (alpha={enet_cv.alpha_:.5f}, l1={enet_cv.l1_ratio_:.2f}): R² = {r2_enet:.4f}   MAE = {mae_enet:.4f}   features kept = {n_enet_kept}/{len(feature_cols)}")

# Which species did Lasso keep?
lasso_coef      = pd.Series(lasso_cv.coef_, index=feature_cols)
lasso_nonzero   = lasso_coef[lasso_coef != 0].sort_values(key=abs, ascending=False)
sp_kept         = int((lasso_coef[species_cols] != 0).sum())
r0_kept         = int((lasso_coef[r0_cols] != 0).sum())
print(f"\nLasso feature selection: Species kept={sp_kept}/100   R0 kept={r0_kept}/20")
print("Lasso top 10 features by |coefficient|:")
for feat, coef in lasso_nonzero.head(10).items():
    ftype = ("Species" if feat in species_cols else "R0" if feat in r0_cols
             else "Active" if feat in active_cols else "Structural")
    print(f"  {feat:<22}  coef={coef:>8.5f}  [{ftype}]")

Baseline comparisons
Baseline (global mean predictor):  R² = -0.0002
Ridge (alpha=30.53856):         R² = 0.8531   MAE = 0.0988
Lasso (alpha=0.00019):       R² = 0.8533   MAE = 0.0986   features kept = 118/142
ElasticNet (alpha=0.00039, l1=0.50): R² = 0.8533   MAE = 0.0986   features kept = 118/142

Lasso feature selection: Species kept=87/100   R0 kept=15/20
Lasso top 10 features by |coefficient|:
  R0_c11                  coef=-0.17510  [R0]
  R0_c17                  coef=-0.13806  [R0]
  R0_c1                   coef=-0.13762  [R0]
  R0_c7                   coef= 0.11975  [R0]
  active_c9               coef= 0.10084  [Active]
  R0_c4                   coef=-0.10037  [R0]
  active_c15              coef=-0.08883  [Active]
  active_c11              coef= 0.08483  [Active]
  active_c5               coef= 0.08177  [Active]
  active_c4               coef=-0.07002  [Active]


In [9]:
# xgboost

model_init = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    early_stopping_rounds=30,
    verbosity=0
)


model_init.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_init = model_init.predict(X_test)
r2_init = r2_score(y_test, y_pred_init)
mae_init = mean_absolute_error(y_test, y_pred_init)
mse_init = mean_squared_error(y_test, y_pred_init)

print(f"XGBoost (initial):  R² = {r2_init:.4f}   MAE = {mae_init:.4f}   MSE = {mse_init:.5f}")
print(f"Best iteration: {model_init.best_iteration}")

XGBoost (initial):  R² = 0.9424   MAE = 0.0478   MSE = 0.00825
Best iteration: 499


In [10]:
# hyperparameter tuning

param_grid = {                          # winning hyperparams from first run used
    'max_depth':        [7],
    
    'learning_rate':    [0.1],
    
    'subsample':        [0.8],
    
    
    'colsample_bytree': [1.0],
   
}

# KFold(n_splits=3): splits training data into 3 folds.
# shuffle=True: randomises row order before splitting
cv = KFold(n_splits=3, shuffle=True, random_state=42)

# base estimator with fixed params for gridsearch
# n_estimators low here to keep it fast, will retrain again with higher value once 
# best params found
base_estimator = xgb.XGBRegressor(
    n_estimators=200,
    min_child_weight=3,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    verbosity=0
)

grid_search = GridSearchCV(
    estimator=base_estimator,
    param_grid=param_grid,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    refit=True
)

# .fit on a GridSearchCV trains ALL combinations and picks the best.
grid_search.fit(X_train, y_train)

print(f"\nBest parameters found:")

for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")


best_cv_rmse = np.sqrt(-grid_search.best_score_)
print(f"Best CV RMSE: {best_cv_rmse:.4f}")

Fitting 3 folds for each of 1 candidates, totalling 3 fits

Best parameters found:
  colsample_bytree: 1.0
  learning_rate: 0.1
  max_depth: 7
  subsample: 0.8
Best CV RMSE: 0.0914


In [11]:
best_params = grid_search.best_params_

model_final = xgb.XGBRegressor(
    **best_params,
    n_estimators=1000,
    min_child_weight=3,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    early_stopping_rounds=40,
    verbosity=0
)

# Pass TWO eval sets: training first, test second.
# This records both training loss and validation loss per round,
# which we need to draw the training curve.
# validation_0 = train loss, validation_1 = test (validation) loss.
model_final.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50
)

y_pred_final = model_final.predict(X_test)
r2_final     = r2_score(y_test, y_pred_final)
mae_final    = mean_absolute_error(y_test, y_pred_final)
mse_final    = mean_squared_error(y_test, y_pred_final)
rmse_final   = np.sqrt(mse_final)

print(f"\nFinal Model Performance")
print(f"R²:   {r2_final:.4f}")
print(f"MAE:  {mae_final:.4f}")
print(f"RMSE: {rmse_final:.4f}")
print(f"MSE:  {mse_final:.5f}")
print(f"Best iteration (trees used): {model_final.best_iteration}")

print("\nModel Comparison")
print(f"{'Model':<30} {'R²':>8}  {'MAE':>8}")
print("-" * 48)
print(f"{'Global mean (baseline)':<30} {r2_mean:>8.4f}  {'N/A':>8}")
print(f"{'Ridge Regression':<30} {r2_ridge:>8.4f}  {mae_ridge:>8.4f}")
print(f"{'Lasso':<30} {r2_lasso:>8.4f}  {mae_lasso:>8.4f}")
print(f"{'Elastic Net':<30} {r2_enet:>8.4f}  {mae_enet:>8.4f}")
print(f"{'XGBoost (initial)':<30} {r2_init:>8.4f}  {mae_init:>8.4f}")
print(f"{'XGBoost (tuned)':<30} {r2_final:>8.4f}  {mae_final:>8.4f}")

# training curve
evals    = model_final.evals_result_
tr_rmse  = evals['validation_0']['rmse']   # training loss per round
val_rmse = evals['validation_1']['rmse']   # validation loss per round
rounds   = range(1, len(tr_rmse) + 1)
best_it  = model_final.best_iteration

fig_tc, ax_tc = plt.subplots(figsize=(9, 4))
ax_tc.plot(rounds, tr_rmse,  color='steelblue',  lw=1.5, label='Train RMSE')
ax_tc.plot(rounds, val_rmse, color='darkorange', lw=1.5, label='Validation RMSE')
ax_tc.axvline(best_it + 1, color='red', linestyle='--', lw=1.2,
              label=f'Best iteration = {best_it + 1}')
ax_tc.set_xlabel('Boosting round')
ax_tc.set_ylabel('RMSE')
ax_tc.set_title(f'XGBoost Training Curve  --  best iter={best_it + 1},  val RMSE={val_rmse[best_it]:.4f}')
ax_tc.legend(fontsize=9)
ax_tc.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('xgb_training_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: xgb_training_curve.png")

# groupkfold

gkf          = GroupKFold(n_splits=5)
fold_r2, fold_mae, fold_rmse = [], [], []

print("\nGroupKFold evaluation (held-out broths):")
for fold_num, (tr_idx, te_idx) in enumerate(
        gkf.split(X, y, groups=groups), start=1):

    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

    held_broths = sorted(set(df.iloc[te_idx]['broth_id']))
    held_divs   = sorted(df.iloc[te_idx]['diversity_level'].unique())
    print(f"\nFold {fold_num}: broths={held_broths}  diversity levels={held_divs}  n_test={len(X_te):,}")

    m_fold = xgb.XGBRegressor(
        **best_params,
        n_estimators=1000, min_child_weight=3, reg_lambda=1.0,
        random_state=42, n_jobs=-1,
        eval_metric='rmse', early_stopping_rounds=40, verbosity=0
    )
    m_fold.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)

    yp_fold  = m_fold.predict(X_te)
    r2_f     = r2_score(y_te, yp_fold)
    mae_f    = mean_absolute_error(y_te, yp_fold)
    rmse_f   = np.sqrt(mean_squared_error(y_te, yp_fold))

    fold_r2.append(r2_f); fold_mae.append(mae_f); fold_rmse.append(rmse_f)
    print(f"  R²={r2_f:.4f}  MAE={mae_f:.4f}  RMSE={rmse_f:.4f}")

print(f"\nGroupKFold summary across {gkf.n_splits} folds:")
print(f"  R²:   mean={np.mean(fold_r2):.4f}  std={np.std(fold_r2):.4f}")
print(f"  MAE:  mean={np.mean(fold_mae):.4f}  std={np.std(fold_mae):.4f}")
print(f"  RMSE: mean={np.mean(fold_rmse):.4f}  std={np.std(fold_rmse):.4f}")
print(f"\nCompare to stratified-split R²={r2_final:.4f}")
print(f"Drop: {r2_final - np.mean(fold_r2):.4f}  (how much was broth memorisation)")

[0]	validation_0-rmse:0.34607	validation_1-rmse:0.34814
[50]	validation_0-rmse:0.10830	validation_1-rmse:0.11164
[100]	validation_0-rmse:0.08754	validation_1-rmse:0.09715
[150]	validation_0-rmse:0.07438	validation_1-rmse:0.09125
[200]	validation_0-rmse:0.06526	validation_1-rmse:0.08840
[250]	validation_0-rmse:0.05754	validation_1-rmse:0.08646
[300]	validation_0-rmse:0.05137	validation_1-rmse:0.08530
[350]	validation_0-rmse:0.04615	validation_1-rmse:0.08489
[400]	validation_0-rmse:0.04169	validation_1-rmse:0.08462
[450]	validation_0-rmse:0.03831	validation_1-rmse:0.08438
[500]	validation_0-rmse:0.03508	validation_1-rmse:0.08417
[550]	validation_0-rmse:0.03217	validation_1-rmse:0.08410
[600]	validation_0-rmse:0.02964	validation_1-rmse:0.08401
[632]	validation_0-rmse:0.02822	validation_1-rmse:0.08405

Final Model Performance
R²:   0.9508
MAE:  0.0438
RMSE: 0.0840
MSE:  0.00705
Best iteration (trees used): 592

Model Comparison
Model                                R²       MAE
------------

In [12]:
print("Performance by diversity level and community size")

results_df = pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          y_pred_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
})

def condition_metrics(y_true_arr, y_pred_arr):
    y_t      = np.asarray(y_true_arr)
    y_p      = np.asarray(y_pred_arr)
    r2       = r2_score(y_t, y_p)
    mae      = mean_absolute_error(y_t, y_p)
    rmse     = np.sqrt(mean_squared_error(y_t, y_p))
    pr, _    = pearsonr(y_t, y_p)
    sr, _    = spearmanr(y_t, y_p)
    bias     = float(np.mean(y_p - y_t))
    std_true = float(np.std(y_t))
    return {'n': len(y_t), 'r2': r2, 'mae': mae, 'rmse': rmse,
            'pearson_r': pr, 'spearman_r': sr,
            'bias': bias, 'std_true': std_true}

hdr = f"{'':6} {'R²':>7} {'MAE':>7} {'RMSE':>7} {'Pearson':>8} {'Spearman':>9} {'Bias':>8} {'std(y)':>7} {'N':>6}"
print("\nBy diversity level:")
print(hdr)
print("-" * len(hdr))
for d, grp in results_df.groupby('diversity_level'):
    m    = condition_metrics(grp['y_true'], grp['y_pred'])
    flag = "  *" if m['std_true'] < 0.08 else "   "
    print(f"D={d:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}{flag}")
print("  * std(y) < 0.08: R² is unreliable here. Read MAE and Spearman r instead.")

print("\nBy community size:")
print(hdr)
print("-" * len(hdr))
for n, grp in results_df.groupby('community_size'):
    m = condition_metrics(grp['y_true'], grp['y_pred'])
    print(f"N={n:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}")

Performance by diversity level and community size

By diversity level:
            R²     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
D=1     0.9483  0.0504  0.1019   0.9738    0.9161   0.0004  0.4483  3,000   
D=2     0.9552  0.0443  0.0873   0.9773    0.9589  -0.0005  0.4123  2,400   
D=4     0.7787  0.0465  0.0827   0.8853    0.8953   0.0008  0.1757  1,800   
D=8     0.9345  0.0326  0.0515   0.9668    0.9713  -0.0014  0.2012  1,200   
D=16    0.9278  0.0333  0.0481   0.9633    0.9673   0.0000  0.1792  1,200   
  * std(y) < 0.08: R² is unreliable here. Read MAE and Spearman r instead.

By community size:
            R²     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
N=12    0.8884  0.0792  0.1244   0.9426    0.9385  -0.0007  0.3724  1,920
N=16    0.9232  0.0599  0.1021   0.9609    0.9456  -0.0019  0.3684  1,920
N=24 

In [13]:
# built in feature importance


importances = model_final.feature_importances_

importance_series = pd.Series(importances, index=feature_cols).sort_values(ascending=False)

print("\nTop 20 most important features (by gain):")
print(importance_series.head(20).round(5).to_string())

# Summarise importance by feature group
species_imp = importance_series[species_cols].sum()
r0_imp      = importance_series[r0_cols].sum()
active_imp  = importance_series[active_cols].sum()
struct_imp  = importance_series[structural_cols].sum()

print(f"\nImportance grouped by feature type (total):")
print(f"  Species membership:  {species_imp:.4f}  ({100*species_imp:.1f}%)")
print(f"  R0 concentrations:   {r0_imp:.4f}  ({100*r0_imp:.1f}%)")
print(f"  Active flags:        {active_imp:.4f}  ({100*active_imp:.1f}%)")
print(f"  Structural (size/div):{struct_imp:.4f}  ({100*struct_imp:.1f}%)")


Top 20 most important features (by gain):
R0_c4              0.46158
R0_c17             0.22977
R0_c11             0.17747
R0_c1              0.03523
R0_c8              0.00897
R0_c7              0.00868
R0_c19             0.00668
R0_c15             0.00458
R0_c3              0.00348
diversity_level    0.00245
R0_c2              0.00221
R0_c14             0.00205
R0_c16             0.00201
Species_0072       0.00186
Species_0064       0.00177
Species_0071       0.00155
Species_0101       0.00149
Species_0034       0.00147
Species_0086       0.00135
community_size     0.00134

Importance grouped by feature type (total):
  Species membership:  0.0468  (4.7%)
  R0 concentrations:   0.9494  (94.9%)
  Active flags:        0.0000  (0.0%)
  Structural (size/div):0.0038  (0.4%)


In [14]:
# shapley value analysis

print('shap analysis')

explainer = shap.TreeExplainer(model_final)

sample_idx = np.random.default_rng(42).choice(len(X_test), size=2000, replace=False)

X_sample = X_test.iloc[sample_idx]


shap_values = explainer.shap_values(X_sample)

mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)

print("\nTop 20 features by mean |SHAP| value:")
print(shap_importance.head(20).round(5).to_string())

# SHAP by feature group
sp_shap     = shap_importance[species_cols].sum()
r0_shap     = shap_importance[r0_cols].sum()
act_shap    = shap_importance[active_cols].sum()
struct_shap = shap_importance[structural_cols].sum()
total_shap  = sp_shap + r0_shap + act_shap + struct_shap

print(f"\nSHAP importance by feature group (sum of mean |SHAP|):")
print(f"  Species membership:  {sp_shap:.4f}  ({100*sp_shap/total_shap:.1f}%)")
print(f"  R0 concentrations:   {r0_shap:.4f}  ({100*r0_shap/total_shap:.1f}%)")
print(f"  Active flags:        {act_shap:.4f}  ({100*act_shap/total_shap:.1f}%)")
print(f"  Structural:          {struct_shap:.4f}  ({100*struct_shap/total_shap:.1f}%)")

shap analysis

Top 20 features by mean |SHAP| value:
R0_c4             0.11743
R0_c1             0.05364
R0_c11            0.05041
R0_c17            0.04628
community_size    0.02986
R0_c19            0.02334
Species_0064      0.01640
Species_0055      0.01602
R0_c8             0.01489
Species_0071      0.01419
Species_0084      0.01341
R0_c16            0.01266
Species_0086      0.01261
R0_c15            0.01204
R0_c3             0.01064
Species_0053      0.01034
R0_c14            0.01010
R0_c9             0.00909
R0_c12            0.00906
R0_c7             0.00888

SHAP importance by feature group (sum of mean |SHAP|):
  Species membership:  0.2669  (37.8%)
  R0 concentrations:   0.4034  (57.1%)
  Active flags:        0.0000  (0.0%)
  Structural:          0.0358  (5.1%)


In [15]:
# figures

print("\n" + "=" * 60)
print("SECTION 12: Generating plots → suppression_results.png")
print("=" * 60)

# plt.figure(figsize=(width, height)): creates a new figure window.
# figsize is in inches. dpi (dots per inch) controls resolution.
fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle('XGBoost Nutrient-Aware Suppression Predictor — MiCRM v8',
             fontsize=15, fontweight='bold', y=0.98)

# plt.subplot(nrows, ncols, panel_index): creates a subplot in a grid.
# Here we use a 2×3 grid (2 rows, 3 columns), panels numbered 1–6.

# ---- Panel 1: Predicted vs Actual ----
ax1 = fig.add_subplot(2, 3, 1)
# ax.scatter(x, y): plots individual data points.
# alpha=0.3 makes points semi-transparent so density is visible.
# c sets colour, s sets marker size.
ax1.scatter(y_test, y_pred_final, alpha=0.15, s=3, c='steelblue')
# plot a y=x reference line (perfect predictions would lie on this line)
ax1.plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Perfect prediction')
ax1.set_xlabel('True community suppression')
ax1.set_ylabel('Predicted community suppression')
ax1.set_title(f'Predicted vs Actual  (R²={r2_final:.3f})')
ax1.legend(fontsize=8)
ax1.set_xlim(-0.05, 1.05)
ax1.set_ylim(-0.05, 1.05)

# ---- Panel 2: Residuals ----
# residuals = y_true - y_pred
# Ideally should be random noise centred at 0.
# Systematic patterns (e.g. always under-predicting at high suppression)
# suggest the model is missing something.
residuals = y_test.values - y_pred_final
ax2 = fig.add_subplot(2, 3, 2)
ax2.scatter(y_pred_final, residuals, alpha=0.15, s=3, c='darkorange')
ax2.axhline(0, color='r', linestyle='--', linewidth=1.5)
# np.percentile(array, [5, 95]): returns the 5th and 95th percentile values.
p5, p95 = np.percentile(residuals, [5, 95])
ax2.axhline(p5,  color='grey', linestyle=':', linewidth=1, label=f'5th pct: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', linewidth=1, label=f'95th pct: {p95:.3f}')
ax2.set_xlabel('Predicted suppression')
ax2.set_ylabel('Residual (true − predicted)')
ax2.set_title('Residual Plot')
ax2.legend(fontsize=7)

# ---- Panel 3: Top 20 SHAP features ----
ax3 = fig.add_subplot(2, 3, 3)
top20_shap = shap_importance.head(20)
colours = []
for feat in top20_shap.index:
    if feat in species_cols:  colours.append('#2196F3')   # blue = species
    elif feat in r0_cols:     colours.append('#4CAF50')   # green = R0
    elif feat in active_cols: colours.append('#FF9800')   # orange = active
    else:                     colours.append('#9C27B0')   # purple = structural
# barh: horizontal bar chart. top20_shap.values = bar lengths.
# [::-1] reverses so highest is at top.
ax3.barh(range(20), top20_shap.values[::-1], color=colours[::-1])
ax3.set_yticks(range(20))
ax3.set_yticklabels(top20_shap.index[::-1], fontsize=7)
ax3.set_xlabel('Mean |SHAP value|')
ax3.set_title('Top 20 Features (SHAP)')

# Add a legend for colour coding
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2196F3', label='Species'),
    Patch(facecolor='#4CAF50', label='R0'),
    Patch(facecolor='#FF9800', label='Active'),
    Patch(facecolor='#9C27B0', label='Structural'),
]
ax3.legend(handles=legend_elements, loc='lower right', fontsize=7)

# ---- Panel 4: R² by diversity level ----
ax4 = fig.add_subplot(2, 3, 4)
div_levels = sorted(results_df['diversity_level'].unique())
r2_by_div = [r2_score(
    results_df[results_df['diversity_level']==d]['y_true'],
    results_df[results_df['diversity_level']==d]['y_pred']
) for d in div_levels]
ax4.bar([str(d) for d in div_levels], r2_by_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', linewidth=1.5, label=f'Overall R²={r2_final:.3f}')
ax4.set_xlabel('Diversity level (# active nutrients)')
ax4.set_ylabel('R²')
ax4.set_title('Model R² by Diversity Level')
ax4.set_ylim(0, 1.05)
ax4.legend(fontsize=8)

# ---- Panel 5: R² by community size ----
ax5 = fig.add_subplot(2, 3, 5)
sizes = sorted(results_df['community_size'].unique())
r2_by_size = [r2_score(
    results_df[results_df['community_size']==n]['y_true'],
    results_df[results_df['community_size']==n]['y_pred']
) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_by_size, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', linewidth=1.5, label=f'Overall R²={r2_final:.3f}')
ax5.set_xlabel('Community size N')
ax5.set_ylabel('R²')
ax5.set_title('Model R² by Community Size')
ax5.set_ylim(0, 1.05)
ax5.legend(fontsize=8)

# ---- Panel 6: SHAP summary by feature group (pie) ----
ax6 = fig.add_subplot(2, 3, 6)
group_names   = ['Species\nmembership', 'R0\nconcentrations', 'Active\nflags', 'Structural']
group_shap    = [sp_shap, r0_shap, act_shap, struct_shap]
group_colours = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
# ax.pie: creates a pie chart.
# autopct='%.1f%%': formats each slice's label as a percentage with 1 decimal.
# startangle: rotates the starting position.
wedges, texts, autotexts = ax6.pie(
    group_shap, labels=group_names, colors=group_colours,
    autopct='%.1f%%', startangle=90, textprops={'fontsize': 9}
)
ax6.set_title('SHAP Importance by Feature Group')

plt.tight_layout(rect=[0, 0, 1, 0.97])
# tight_layout: automatically adjusts subplot spacing to avoid overlap.
# rect=[left, bottom, right, top] reserves space for the suptitle.

plt.savefig('suppression_results_updated02.png', dpi=150, bbox_inches='tight')
# bbox_inches='tight': crops the saved figure to the actual content boundary
# (removes whitespace margins). Always use this.
plt.close()
# plt.close(): frees memory by closing the figure after saving.

print("Saved: suppression_results_updated02.png")


SECTION 12: Generating plots → suppression_results.png
Saved: suppression_results_updated02.png


In [ ]:
# =============================================================================
# SECTION 13 — SAVE MODEL & PREDICTIONS
# =============================================================================

print("\n" + "=" * 60)
print("SECTION 13: Saving outputs")
print("=" * 60)

# Save the model in XGBoost's native JSON format.
# This preserves all tree structures and can be reloaded with:
#   loaded_model = xgb.XGBRegressor()
#   loaded_model.load_model('xgb_model_v8.json')
model_final.save_model('xgb_model_v8.json')
print("Model saved: xgb_model_v8.json")

# Save predictions alongside true values for downstream analysis.
pred_df = pd.DataFrame({
    'community_id':       df.loc[X_test.index, 'community_id'].values,
    'broth_id':           df.loc[X_test.index, 'broth_id'].values,
    'diversity_level':    df.loc[X_test.index, 'diversity_level'].values,
    'community_size':     df.loc[X_test.index, 'community_size'].values,
    'y_true':             y_test.values,
    'y_pred':             y_pred_final,
    'residual':           y_test.values - y_pred_final,
})
pred_df.to_csv('predictions_v8.csv', index=False)
print("Predictions saved: predictions_v8.csv")

# Save SHAP values for the 2000-sample subset.
shap_df = pd.DataFrame(shap_values, columns=feature_cols)
shap_df.to_csv('shap_values_v8.csv', index=False)
print("SHAP values saved: shap_values_v8.csv")